# HOG/SVM convencional por janela deslizante usando o modelo V2

Este notebook usa o **modelo HOG/SVM já treinado na abordagem V2 customizada** e altera apenas a forma de gerar candidatos: em vez de pares de retas, usa **janela deslizante**.

Objetivo: comparar a quantidade de regiões avaliadas, tempo de detecção e métricas com o último resultado salvo do V2 interativo em:

`vision/results/classical_cylinder_detector/historico_treinos_v2.csv`

In [ ]:
from pathlib import Path
import sys, os, json, time, math, random
import numpy as np
import pandas as pd
import cv2
from joblib import load
from skimage.feature import hog
from IPython.display import display, Markdown
import ipywidgets as widgets

def encontrar_raiz_blaze(start=None):
    p = Path(start or Path.cwd()).resolve()
    candidatos = [p] + list(p.parents)
    for c in candidatos:
        if (c / "src" / "blaze_paths.py").exists() and (c / "vision").exists():
            return c
    for c in candidatos:
        if (c / "vision").exists():
            return c
    raise RuntimeError("Não consegui encontrar a raiz do BLAZE. Abra o notebook dentro da pasta do projeto.")

PROJECT_ROOT = encontrar_raiz_blaze()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from blaze_paths import (
        VISION_DATASETS_DIR,
        VISION_RESULTS_DIR,
        VISION_USER_SETUPS_DIR,
        VISION_OFFICIAL_SETUPS_DIR,
        ensure_base_dirs,
    )
    ensure_base_dirs()
except Exception as e:
    print("Aviso: não consegui importar blaze_paths.py. Usando caminhos relativos ao PROJECT_ROOT.")
    VISION_DATASETS_DIR = PROJECT_ROOT / "vision" / "datasets"
    VISION_RESULTS_DIR = PROJECT_ROOT / "vision" / "results"
    VISION_USER_SETUPS_DIR = PROJECT_ROOT / "vision" / "parameters_setups" / "user"
    VISION_OFFICIAL_SETUPS_DIR = PROJECT_ROOT / "vision" / "parameters_setups" / "official"

DATASET_DIR = VISION_DATASETS_DIR / "cylinders" / "CylinDeRS-1"
RESULTS_DIR = VISION_RESULTS_DIR / "classical_cylinder_detector"
USER_SETUP_DIR = VISION_USER_SETUPS_DIR / "cylinders_detect"
OFFICIAL_SETUP_DIR = VISION_OFFICIAL_SETUPS_DIR / "cylinders_detect"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
USER_SETUP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_V2_PATH = RESULTS_DIR / "modelo_hog_svm_cilindros_v2.joblib"
METADATA_V2_PATH = RESULTS_DIR / "metadata_modelo_v2.json"
HIST_V2_PATH = RESULTS_DIR / "historico_treinos_v2.csv"

SETUP_CONV_PATHS = [
    USER_SETUP_DIR / "setups_parametricos_convencional_modelo_v2_compara_corrigido_user.json",
    USER_SETUP_DIR / "setups_parametricos_convencional_modelo_v2_user.json",
    USER_SETUP_DIR / "setups_parametricos_convencional_sliders_user.json",
]
HIST_CONV_PATH = RESULTS_DIR / "historico_treinos_convencional_modelo_v2_compara_corrigido.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("MODEL_V2_PATH:", MODEL_V2_PATH)
print("HIST_V2_PATH:", HIST_V2_PATH)

for nome, caminho in [
    ("Dataset", DATASET_DIR),
    ("Modelo V2", MODEL_V2_PATH),
    ("Histórico V2", HIST_V2_PATH),
]:
    print(f"{nome}: {'OK' if caminho.exists() else 'NÃO ENCONTRADO'} -> {caminho}")

## Carregar setup convencional e último resultado V2

A célula abaixo procura o setup convencional nos caminhos corrigidos de `vision/parameters_setups/user/cylinders_detect/`.

Se o JSON não existir, ela cria um setup padrão usando os parâmetros do último V2 salvo, quando disponíveis.

In [ ]:
def carregar_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def salvar_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def carregar_ultimo_v2():
    hist = None
    row = None
    if HIST_V2_PATH.exists():
        hist = pd.read_csv(HIST_V2_PATH)
        if len(hist) > 0:
            row = hist.iloc[-1].to_dict()
    return hist, row

hist_v2, row_v2 = carregar_ultimo_v2()
if row_v2:
    print("Último resultado V2 encontrado:")
    print("  setup_id:", row_v2.get("setup_id"))
    print("  setup_nome:", row_v2.get("setup_nome"))
    print("  data_hora:", row_v2.get("data_hora"))
    print("  det_iou_thr:", row_v2.get("det_iou_thr"))
    print("  f1_det:", row_v2.get("f1_det"))
else:
    print("Histórico V2 não encontrado ou vazio. A comparação final ficará sem a linha customizada.")

def params_padrao_a_partir_v2(row_v2=None):
    # Valores seguros e compatíveis com o V2 robusto; podem ser alterados nos widgets.
    return {
        "clahe_ativo": True,
        "clahe_clip": 1.5,
        "clahe_grid": 16,
        "bilateral_d": 5,
        "bilateral_sigma_color": 60,
        "bilateral_sigma_space": 50,
        "canny_sigma": 0.33,
        "canny_aperture": 3,
        "canny_l2gradient": True,
        "hog_input": "bilateral",
        "hog_resize_w": int(row_v2.get("hog_size", "64x160").split("x")[0]) if row_v2 and isinstance(row_v2.get("hog_size"), str) else 64,
        "hog_resize_h": int(row_v2.get("hog_size", "64x160").split("x")[1]) if row_v2 and isinstance(row_v2.get("hog_size"), str) else 160,
        "hog_orientations": int(row_v2.get("hog_orientations", 16)) if row_v2 else 16,
        "hog_pixels_per_cell": int(row_v2.get("hog_cell", 4)) if row_v2 else 4,
        "hog_cells_per_block": int(row_v2.get("hog_block", 2)) if row_v2 else 2,
        "slide_win_w": 64,
        "slide_win_h": 160,
        "slide_scales": "0.75,1.0,1.25,1.5,2.0",
        "slide_step_px": 32,
        "slide_max_windows": 5000,
        "det_score_min": float(row_v2.get("det_score_min", 0.0)) if row_v2 else 0.0,
        "det_nms_iou": float(row_v2.get("det_nms_iou", 0.40)) if row_v2 else 0.40,
        "det_iou_thr": float(row_v2.get("det_iou_thr", 0.25)) if row_v2 else 0.25,
        "det_max_det": int(row_v2.get("det_max_det", 15)) if row_v2 else 15,
        "det_eval_max_images": 30,
    }

def carregar_setup_convencional():
    for p in SETUP_CONV_PATHS:
        if p.exists():
            data = carregar_json(p)
            setups = data.get("setups", {})
            if "CONV_V2_COMP" in setups:
                return setups["CONV_V2_COMP"]["params"], "CONV_V2_COMP", p
            if setups:
                sid = list(setups.keys())[-1]
                return setups[sid]["params"], sid, p

    params = params_padrao_a_partir_v2(row_v2)
    out = {
        "versao": "setups_parametricos_convencional_modelo_v2_compara_corrigido",
        "descricao": "Setup criado automaticamente: janela deslizante usando modelo V2.",
        "setups": {
            "CONV_V2_COMP": {
                "nome": "Convencional janela deslizante | mesmo modelo V2 | compara com último V2",
                "data_hora": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
                "params": params,
            }
        },
    }
    salvar_json(SETUP_CONV_PATHS[0], out)
    return params, "CONV_V2_COMP", SETUP_CONV_PATHS[0]

params, setup_id, setup_path = carregar_setup_convencional()
print("Setup convencional carregado:", setup_id)
print("Arquivo:", setup_path)

print("\nParâmetros principais:")
for k in ["hog_input", "hog_resize_w", "hog_resize_h", "hog_orientations", "hog_pixels_per_cell",
          "hog_cells_per_block", "slide_scales", "slide_step_px", "slide_max_windows",
          "det_score_min", "det_nms_iou", "det_iou_thr", "det_eval_max_images"]:
    print(f"  {k}: {params.get(k)}")

## Ajustar carga da janela deslizante

Esses controles afetam apenas a busca convencional por janelas. O modelo HOG/SVM usado é o V2 já treinado.

In [ ]:
w_scales = widgets.Text(
    value=str(params.get("slide_scales", "0.75,1.0,1.25,1.5,2.0")),
    description="escalas",
    layout=widgets.Layout(width="420px")
)
w_step = widgets.IntSlider(value=int(params.get("slide_step_px", 32)), min=8, max=96, step=4, description="passo px")
w_max_windows = widgets.IntSlider(value=int(params.get("slide_max_windows", 5000)), min=500, max=30000, step=500, description="max janelas")
w_eval_imgs = widgets.IntSlider(value=int(params.get("det_eval_max_images", 30)), min=5, max=310, step=5, description="imgs aval.")
w_score = widgets.FloatSlider(value=float(params.get("det_score_min", 0.0)), min=-10, max=10, step=0.1, description="score mín")
w_nms = widgets.FloatSlider(value=float(params.get("det_nms_iou", 0.4)), min=0.05, max=0.9, step=0.05, description="NMS IoU")
w_iou = widgets.FloatSlider(value=float(params.get("det_iou_thr", 0.25)), min=0.05, max=0.75, step=0.05, description="IoU acerto")
w_max_det = widgets.IntSlider(value=int(params.get("det_max_det", 15)), min=1, max=50, step=1, description="max det")

display(widgets.VBox([
    widgets.HTML("<b>Parâmetros da busca por janela deslizante</b>"),
    w_scales,
    widgets.HBox([w_step, w_max_windows]),
    widgets.HBox([w_eval_imgs, w_score]),
    widgets.HBox([w_nms, w_iou, w_max_det]),
]))

def atualizar_params_widgets():
    params["slide_scales"] = w_scales.value
    params["slide_step_px"] = int(w_step.value)
    params["slide_max_windows"] = int(w_max_windows.value)
    params["det_eval_max_images"] = int(w_eval_imgs.value)
    params["det_score_min"] = float(w_score.value)
    params["det_nms_iou"] = float(w_nms.value)
    params["det_iou_thr"] = float(w_iou.value)
    params["det_max_det"] = int(w_max_det.value)
    return params

## Funções de detecção convencional

Esta seção carrega o modelo V2 e aplica janela deslizante no conjunto de teste.

In [ ]:
def listar_imagens_teste(dataset_dir):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp"]
    paths = []
    for sub in ["test/images", "valid/images", "train/images"]:
        d = dataset_dir / sub
        if d.exists():
            for e in exts:
                paths.extend(sorted(d.glob(e)))
            if paths:
                print("Usando imagens em:", d)
                return sorted(paths)
    raise FileNotFoundError(f"Não encontrei imagens em {dataset_dir}/test/images, valid/images ou train/images.")

def label_path_para_imagem(img_path, dataset_dir):
    # troca /images/ por /labels/ e extensão por .txt
    parts = list(img_path.parts)
    if "images" in parts:
        idx = parts.index("images")
        parts[idx] = "labels"
        return Path(*parts).with_suffix(".txt")
    return img_path.with_suffix(".txt")

def carregar_gt_yolo(img_path, dataset_dir):
    img = cv2.imread(str(img_path))
    if img is None:
        return []
    H, W = img.shape[:2]
    label_path = label_path_para_imagem(img_path, dataset_dir)
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text(encoding="utf-8").strip().splitlines():
        if not line.strip():
            continue
        vals = line.split()
        if len(vals) < 5:
            continue
        cls, xc, yc, bw, bh = vals[:5]
        xc, yc, bw, bh = map(float, [xc, yc, bw, bh])
        x1 = int(round((xc - bw / 2) * W))
        y1 = int(round((yc - bh / 2) * H))
        x2 = int(round((xc + bw / 2) * W))
        y2 = int(round((yc + bh / 2) * H))
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W - 1, x2), min(H - 1, y2)
        if x2 > x1 and y2 > y1:
            boxes.append((x1, y1, x2, y2))
    return boxes

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def nms(dets, iou_thr=0.4, max_det=15):
    if not dets:
        return []
    dets = sorted(dets, key=lambda d: d["score"], reverse=True)
    keep = []
    for d in dets:
        if all(iou_xyxy(d["bbox"], k["bbox"]) < iou_thr for k in keep):
            keep.append(d)
        if len(keep) >= max_det:
            break
    return keep

def avaliar_deteccoes(preds, gts, iou_thr):
    matched = set()
    tp = 0
    fp = 0
    best_ious = []
    tp_ious = []
    for p in sorted(preds, key=lambda d: d["score"], reverse=True):
        if gts:
            ious = [iou_xyxy(p["bbox"], gt) for gt in gts]
            best = max(ious)
            j = int(np.argmax(ious))
        else:
            best, j = 0.0, -1
        best_ious.append(best)
        if best >= iou_thr and j not in matched:
            tp += 1
            matched.add(j)
            tp_ious.append(best)
        else:
            fp += 1
    fn = len(gts) - tp
    return tp, fp, fn, best_ious, tp_ious

def preprocessar_para_hog(img_bgr, params):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if params.get("clahe_ativo", True):
        clip = float(params.get("clahe_clip", 1.5))
        grid = int(params.get("clahe_grid", 16))
        clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
        gray = clahe.apply(gray)
    d = int(params.get("bilateral_d", 5))
    sigma_color = float(params.get("bilateral_sigma_color", 60))
    sigma_space = float(params.get("bilateral_sigma_space", 50))
    if d > 0:
        bilateral = cv2.bilateralFilter(gray, d, sigma_color, sigma_space)
    else:
        bilateral = gray
    return bilateral if params.get("hog_input", "bilateral") == "bilateral" else gray

def extrair_hog_roi(gray_img, bbox, params):
    x1, y1, x2, y2 = map(int, bbox)
    roi = gray_img[max(0,y1):max(0,y2), max(0,x1):max(0,x2)]
    if roi.size == 0:
        return None
    w = int(params.get("hog_resize_w", 64))
    h = int(params.get("hog_resize_h", 160))
    roi = cv2.resize(roi, (w, h), interpolation=cv2.INTER_AREA)
    feat = hog(
        roi,
        orientations=int(params.get("hog_orientations", 16)),
        pixels_per_cell=(int(params.get("hog_pixels_per_cell", 4)), int(params.get("hog_pixels_per_cell", 4))),
        cells_per_block=(int(params.get("hog_cells_per_block", 2)), int(params.get("hog_cells_per_block", 2))),
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True,
    )
    return np.asarray(feat, dtype=np.float32)

def score_modelo(clf, feat):
    X = feat.reshape(1, -1)
    if hasattr(clf, "decision_function"):
        s = clf.decision_function(X)
        return float(np.ravel(s)[0])
    if hasattr(clf, "predict_proba"):
        return float(clf.predict_proba(X)[0, 1])
    pred = clf.predict(X)[0]
    return float(pred)

def gerar_janelas(H, W, params):
    base_w = int(params.get("slide_win_w", params.get("hog_resize_w", 64)))
    base_h = int(params.get("slide_win_h", params.get("hog_resize_h", 160)))
    scales = [float(s.strip()) for s in str(params.get("slide_scales", "1.0")).split(",") if s.strip()]
    step = int(params.get("slide_step_px", 32))
    max_windows = int(params.get("slide_max_windows", 5000))
    count = 0
    for scale in scales:
        ww = max(8, int(round(base_w * scale)))
        wh = max(8, int(round(base_h * scale)))
        if ww >= W or wh >= H:
            continue
        for y in range(0, H - wh + 1, step):
            for x in range(0, W - ww + 1, step):
                yield (x, y, x + ww, y + wh)
                count += 1
                if max_windows > 0 and count >= max_windows:
                    return

def detectar_janela_deslizante(img_bgr, clf, params):
    gray_hog = preprocessar_para_hog(img_bgr, params)
    H, W = gray_hog.shape[:2]
    score_min = float(params.get("det_score_min", 0.0))
    dets = []
    n_windows = 0
    for bbox in gerar_janelas(H, W, params):
        n_windows += 1
        feat = extrair_hog_roi(gray_hog, bbox, params)
        if feat is None:
            continue
        score = score_modelo(clf, feat)
        if score >= score_min:
            dets.append({"bbox": bbox, "score": score})
    dets_nms = nms(dets, float(params.get("det_nms_iou", 0.4)), int(params.get("det_max_det", 15)))
    return dets_nms, n_windows, len(dets)

## Rodar avaliação com janela deslizante usando o modelo V2

A célula abaixo pode demorar, porque a janela deslizante avalia muitas regiões por imagem.

In [ ]:
params = atualizar_params_widgets()

if not MODEL_V2_PATH.exists():
    raise FileNotFoundError(f"Modelo V2 não encontrado: {MODEL_V2_PATH}")

clf = load(MODEL_V2_PATH)
print("Modelo V2 carregado:", MODEL_V2_PATH)

imagens = listar_imagens_teste(DATASET_DIR)
max_imgs = int(params.get("det_eval_max_images", 30))
if max_imgs > 0:
    imagens = imagens[:max_imgs]

print(f"Imagens avaliadas: {len(imagens)}")
print("Parâmetros de janela:", {
    "slide_scales": params.get("slide_scales"),
    "slide_step_px": params.get("slide_step_px"),
    "slide_max_windows": params.get("slide_max_windows"),
    "score_min": params.get("det_score_min"),
    "iou_acerto": params.get("det_iou_thr"),
})

t0 = time.perf_counter()
TP = FP = FN = 0
n_gt_total = 0
n_pred_total = 0
total_windows = 0
total_pre_nms = 0
best_ious_all = []
tp_ious_all = []

for idx, img_path in enumerate(imagens, start=1):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    gts = carregar_gt_yolo(img_path, DATASET_DIR)
    preds, n_windows, n_pre_nms = detectar_janela_deslizante(img, clf, params)
    tp, fp, fn, best_ious, tp_ious = avaliar_deteccoes(preds, gts, float(params.get("det_iou_thr", 0.25)))
    TP += tp; FP += fp; FN += fn
    n_gt_total += len(gts)
    n_pred_total += len(preds)
    total_windows += n_windows
    total_pre_nms += n_pre_nms
    best_ious_all.extend(best_ious)
    tp_ious_all.extend(tp_ious)
    if idx % 5 == 0 or idx == len(imagens):
        print(f"[{idx}/{len(imagens)}] janelas={total_windows} pred={n_pred_total} TP={TP} FP={FP} FN={FN}")

tempo_s = time.perf_counter() - t0
precision_det = TP / (TP + FP) if (TP + FP) else 0.0
recall_det = TP / (TP + FN) if (TP + FN) else 0.0
f1_det = 2 * precision_det * recall_det / (precision_det + recall_det) if (precision_det + recall_det) else 0.0

row_conv = {
    "data_hora": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "metodo": "convencional_janela_deslizante_modelo_v2",
    "setup_id": setup_id,
    "setup_nome": "Janela deslizante usando modelo HOG/SVM V2",
    "modelo_usado": str(MODEL_V2_PATH),
    "det_eval_imgs": len(imagens),
    "n_gt_det": n_gt_total,
    "n_pred_det": n_pred_total,
    "n_windows_total": total_windows,
    "n_windows_per_img": total_windows / len(imagens) if imagens else np.nan,
    "n_pre_nms": total_pre_nms,
    "det_iou_thr": float(params.get("det_iou_thr", 0.25)),
    "mean_iou_best": float(np.mean(best_ious_all)) if best_ious_all else 0.0,
    "median_iou_best": float(np.median(best_ious_all)) if best_ious_all else 0.0,
    "mean_iou_tp": float(np.mean(tp_ious_all)) if tp_ious_all else 0.0,
    "precision_det": precision_det,
    "recall_det": recall_det,
    "f1_det": f1_det,
    "tp_det": TP,
    "fp_det": FP,
    "fn_det": FN,
    "tempo_s": tempo_s,
    "tempo_por_img_s": tempo_s / len(imagens) if imagens else np.nan,
    "hog_input": params.get("hog_input"),
    "hog_size": f"{params.get('hog_resize_w')}x{params.get('hog_resize_h')}",
    "hog_orientations": int(params.get("hog_orientations", 16)),
    "hog_cell": int(params.get("hog_pixels_per_cell", 4)),
    "hog_block": int(params.get("hog_cells_per_block", 2)),
    "slide_scales": params.get("slide_scales"),
    "slide_step_px": int(params.get("slide_step_px", 32)),
    "slide_max_windows": int(params.get("slide_max_windows", 5000)),
    "det_score_min": float(params.get("det_score_min", 0.0)),
    "det_nms_iou": float(params.get("det_nms_iou", 0.4)),
    "det_max_det": int(params.get("det_max_det", 15)),
    "dataset_path": str(DATASET_DIR),
}

hist_new = pd.DataFrame([row_conv])
if HIST_CONV_PATH.exists():
    old = pd.read_csv(HIST_CONV_PATH)
    hist_out = pd.concat([old, hist_new], ignore_index=True)
else:
    hist_out = hist_new
hist_out.to_csv(HIST_CONV_PATH, index=False)

print("\nAvaliação convencional concluída.")
print("Histórico salvo em:", HIST_CONV_PATH)
display(pd.DataFrame([row_conv]).T)

## Comparar com o último resultado da V2 interativa

Esta célula usa:

- Customizado V2: `historico_treinos_v2.csv`
- Convencional: `historico_treinos_convencional_modelo_v2_compara_corrigido.csv`

In [ ]:
def ultimo_registro(path):
    if not Path(path).exists:
        return None
    df = pd.read_csv(path)
    if len(df) == 0:
        return None
    return df.iloc[-1].to_dict()

row_v2 = ultimo_registro(HIST_V2_PATH)
row_conv = ultimo_registro(HIST_CONV_PATH)

linhas = []
if row_v2:
    linhas.append({
        "método": "Customizado V2 — pares de retas",
        "setup": row_v2.get("setup_id", ""),
        "imgs aval.": row_v2.get("det_eval_imgs", np.nan),
        "regiões avaliadas": row_v2.get("n_pred_det", np.nan),
        "regiões/img": (row_v2.get("n_pred_det", np.nan) / row_v2.get("det_eval_imgs", np.nan)) if row_v2.get("det_eval_imgs", 0) else np.nan,
        "IoU acerto": row_v2.get("det_iou_thr", np.nan),
        "precisão det.": row_v2.get("precision_det", np.nan),
        "recall det.": row_v2.get("recall_det", np.nan),
        "F1 det.": row_v2.get("f1_det", np.nan),
        "TP/FP/FN": f"{int(row_v2.get('tp_det',0))}/{int(row_v2.get('fp_det',0))}/{int(row_v2.get('fn_det',0))}",
        "tempo total (s)": row_v2.get("tempo_s", np.nan),
        "tempo/img (s)": (row_v2.get("tempo_s", np.nan) / row_v2.get("det_eval_imgs", np.nan)) if row_v2.get("det_eval_imgs", 0) else np.nan,
        "observação": "ROIs geométricas; regiões avaliadas = detecções finais registradas no histórico V2",
    })
else:
    print("Histórico V2 não encontrado:", HIST_V2_PATH)

if row_conv:
    linhas.append({
        "método": "Convencional — janela deslizante + modelo V2",
        "setup": row_conv.get("setup_id", ""),
        "imgs aval.": row_conv.get("det_eval_imgs", np.nan),
        "regiões avaliadas": row_conv.get("n_windows_total", np.nan),
        "regiões/img": row_conv.get("n_windows_per_img", np.nan),
        "IoU acerto": row_conv.get("det_iou_thr", np.nan),
        "precisão det.": row_conv.get("precision_det", np.nan),
        "recall det.": row_conv.get("recall_det", np.nan),
        "F1 det.": row_conv.get("f1_det", np.nan),
        "TP/FP/FN": f"{int(row_conv.get('tp_det',0))}/{int(row_conv.get('fp_det',0))}/{int(row_conv.get('fn_det',0))}",
        "tempo total (s)": row_conv.get("tempo_s", np.nan),
        "tempo/img (s)": row_conv.get("tempo_por_img_s", np.nan),
        "observação": "Busca exaustiva; regiões avaliadas = janelas varridas antes do SVM",
    })
else:
    print("Histórico convencional não encontrado:", HIST_CONV_PATH)

comparacao = pd.DataFrame(linhas)
if len(comparacao):
    display(comparacao)

    # Versão compacta arredondada para relatório
    comp_fmt = comparacao.copy()
    for col in ["regiões/img", "IoU acerto", "precisão det.", "recall det.", "F1 det.", "tempo total (s)", "tempo/img (s)"]:
        if col in comp_fmt.columns:
            comp_fmt[col] = pd.to_numeric(comp_fmt[col], errors="coerce").round(4)
    display(Markdown("### Tabela compacta para o relatório"))
    display(comp_fmt)

    if row_v2 and row_conv:
        try:
            reducao = float(row_conv.get("n_windows_per_img", np.nan)) / float(linhas[0]["regiões/img"])
            print(f"\nA janela deslizante avaliou aproximadamente {reducao:.1f} vezes mais regiões por imagem que o V2.")
        except Exception:
            pass

## Observação sobre a comparação

A linha do método convencional contabiliza as **janelas varridas** antes do SVM.  
A linha do V2 customizado usa o histórico do detector de pares de retas. Se o histórico V2 não tiver salvo o número bruto de ROIs antes do HOG/SVM, a tabela usa `n_pred_det` como aproximação conservadora das regiões finais registradas.